# Автокодировщики

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Автокодировщики».

## Научная цель

Автокодировщик — нейронная сеть, которую учат воспроизводить собственный вход. Она состоит из кодировщика, сжимающего объект в компактное описание, и декодировщика, восстанавливающего объект обратно. Проходя через узкое «горлышко», сеть вынуждена выделить в данных главное. Для учёного это способ без разметки найти компактное представление сложных данных, убрать шум и выявить нетипичные объекты.

В этом блокноте мы рассмотрим три применения автокодировщика на компактном наборе изображений цифр: понижение размерности, шумоподавление и поиск аномалий по ошибке восстановления. Блокнот исполняется на CPU за минуту.

## Интуиция за методом

Представьте, что вы конспектируете лекцию. Записать каждое слово невозможно — вы фиксируете суть, а потом по конспекту воспроизводите содержание. Хороший конспект короткий, но информативный: из него теряется несущественное и сохраняется главное.

Автокодировщик делает то же самое с данными. Он состоит из двух частей:

- **Кодировщик** (encoder) — принимает исходный объект (изображение, спектр, сигнал) и «конспектирует» его в короткий набор чисел — **скрытый вектор** (latent vector). Размер этого вектора намного меньше размера исходного объекта.
- **Декодировщик** (decoder) — берёт скрытый вектор и пытается восстановить исходный объект.

Сеть обучается без разметки: её задача — воспроизвести собственный вход. Проходя через узкое «горлышко» (скрытый вектор), она вынуждена выделить в данных самое важное. Всё лишнее — в том числе случайный шум — через горлышко не проходит и теряется.

Полученный скрытый вектор можно использовать трояко:
1. Как **сжатое представление** данных (аналог метода главных компонент, но нелинейный).
2. Для **шумоподавления**: подать зашумлённый объект, получить восстановленный чистый.
3. Для **поиска аномалий**: нетипичный объект автокодировщик восстановит плохо — с большой ошибкой.

Терминологический минимум для этого блокнота:
- **Эпоха** (epoch) — один полный проход обучения через все данные.
- **Функция потерь** (loss function) — число, которое сеть минимизирует. Здесь это ошибка восстановления входа.
- **MSE** (mean squared error) — среднеквадратичная ошибка: среднее квадратов отклонений.

## 1. Установка библиотек

В Google Colab большинство пакетов уже установлено. Ячейка ниже гарантирует наличие нужных библиотек. При локальном запуске она также корректна.

In [ ]:
%pip install -q torch==2.3.1 scikit-learn numpy matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем открытый набор рукописных цифр `digits` из `scikit-learn` (1797 изображений 8x8). Для опыта с поиском аномалий обучим автокодировщик только на цифрах 0–8, а цифру 9 оставим как «нетипичный» класс, не виденный при обучении.

In [ ]:
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

data = load_digits()
X = (data.images.reshape(len(data.images), -1)
     .astype('float32') / 16.0)
y = data.target

# Норма: цифры 0-8. Аномалия: цифра 9 (при обучении не видна).
normal = X[y != 9]
anomaly = X[y == 9]
y_normal = y[y != 9]

X_train, X_test, lbl_train, lbl_test = train_test_split(
    normal, y_normal, test_size=0.2, random_state=0)
print(f'Обучающих (норма): {len(X_train)}, тестовых (норма): {len(X_test)}')
print(f'Аномальных примеров (цифра 9): {len(anomaly)}')

## 4. Применение метода

### Шаг 1. Архитектура автокодировщика

Построим нейронную сеть с «горлышком». Параметр `LATENT` — это ширина горлышка: сколько чисел используется для описания каждого изображения. Исходное изображение — вектор из 64 чисел (8x8 пикселей); мы сжимаем его до `LATENT = 8` чисел, то есть в 8 раз.

Кодировщик последовательно уменьшает размерность: 64 → 32 → 8. Декодировщик делает обратное: 8 → 32 → 64. После декодировщика применяется `Sigmoid`, чтобы все значения лежали в [0, 1] — как и исходные пиксели.

В выводе ниже вы увидите структуру сети — количество слоёв и их размеры.

In [ ]:
import torch.nn as nn

LATENT = 8          # размер «горлышка»


class Autoencoder(nn.Module):
    """Автокодировщик для изображений 8x8 (вектор длины 64)."""

    def __init__(self, n_in=64, latent=LATENT):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_in, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, n_in), nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


model = Autoencoder()
print(model)

### Шаг 2. Обучение автокодировщика

Обучим сеть 60 эпох. На каждой эпохе:
1. Берём пакет (batch) изображений.
2. Прогоняем через кодировщик и декодировщик.
3. Считаем ошибку восстановления (MSE между входом и восстановлением).
4. Обновляем веса сети оптимизатором Adam, чтобы ошибка уменьшалась.

`batch_size = 64` — количество изображений в одном пакете. Оптимизатор Adam адаптивно настраивает шаг обучения для каждого параметра.

Ниже выводится ошибка на каждые 15 эпох. Следите, как она убывает — это признак обучения.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

ds = TensorDataset(torch.from_numpy(X_train))
loader = DataLoader(ds, batch_size=64, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
crit = nn.MSELoss()

history = []
for epoch in range(1, 61):
    model.train()
    ep = 0.0
    for (xb,) in loader:
        opt.zero_grad()
        recon, _ = model(xb)
        loss = crit(recon, xb)        # ошибка восстановления входа
        loss.backward()
        opt.step()
        ep += loss.item() * len(xb)
    history.append(ep / len(X_train))
    if epoch % 15 == 0:
        print(f'Эпоха {epoch:2d}: ошибка восстановления {history[-1]:.5f}')

### Применение 1. Понижение размерности и шумоподавление

**Понижение размерности:** извлечём скрытые векторы (8-мерные) из тестовых изображений и спроецируем их в 2D через метод главных компонент (PCA) — чисто для отображения на плоскости. Если автокодировщик уловил структуру данных, цифры одного класса должны собраться в группы.

**Шумоподавление:** добавим к тестовым изображениям случайный гауссов шум и пропустим через автокодировщик. Горлышко «отфильтрует» шум, восстановив чистые изображения.

Ниже выведутся два графика — объяснение как их читать приводится после ячейки.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

model.eval()
with torch.no_grad():
    _, z_test = model(torch.from_numpy(X_test))
z_test = z_test.numpy()
# Скрытое пространство 8-мерно — для показа проецируем в 2D через PCA.
z_2d = PCA(n_components=2, random_state=0).fit_transform(z_test)

# Шумоподавление: добавим шум к тестовым изображениям.
noisy = np.clip(X_test + 0.25 * np.random.randn(*X_test.shape),
                0, 1).astype('float32')
with torch.no_grad():
    denoised, _ = model(torch.from_numpy(noisy))
denoised = denoised.numpy()

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))
sc = axes[0].scatter(z_2d[:, 0], z_2d[:, 1], c=lbl_test,
                     cmap='tab10', s=22, alpha=0.8,
                     edgecolor='white', linewidth=0.3)
axes[0].set_title('Скрытое представление (проекция в 2D)')
axes[0].set_xlabel('Главная компонента 1')
axes[0].set_ylabel('Главная компонента 2')
fig.colorbar(sc, ax=axes[0], fraction=0.046, pad=0.04,
             label='Класс цифры')

axes[1].plot(range(1, len(history) + 1), history,
             color=VIZ['series'][0])
axes[1].set_title('Обучение автокодировщика')
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Ошибка восстановления')
fig.tight_layout()
plt.show()

**Как читать эти графики:**

- **Левый — скрытое представление (карта данных):** каждая точка — одно тестовое изображение, цвет — класс цифры. Если точки одного цвета образуют плотные кластеры и цвета не перемешиваются — автокодировщик научился отличать разные цифры в скрытом пространстве. Это нелинейная структура, которую PCA нашло бы хуже.
- **Правый — кривая обучения:** по оси X — номер эпохи, по оси Y — ошибка восстановления. Монотонное убывание означает, что сеть обучается успешно. Если кривая «встала» рано — попробуйте увеличить число эпох или уменьшить `LATENT`.

Обратите внимание: граница между цифрами 4, 7, 9 обычно размытее, чем между 0 и 1 — это отражает реальную визуальную похожесть рукописных цифр.

In [ ]:
# Шумоподавление: верх — зашумлённый вход, низ — восстановление сети.
fig, axes = plt.subplots(2, 8, figsize=(12, 3.6))
for k in range(8):
    axes[0, k].imshow(noisy[k].reshape(8, 8), cmap='gray_r')
    axes[1, k].imshow(denoised[k].reshape(8, 8), cmap='gray_r')
    for r in (0, 1):
        axes[r, k].set_xticks([]); axes[r, k].set_yticks([])
        axes[r, k].grid(False)
axes[0, 0].set_ylabel('Зашумлённый\nвход', fontsize=11)
axes[1, 0].set_ylabel('Восстановление\nсети', fontsize=11)
fig.suptitle('Шумоподавление автокодировщиком', fontsize=13)
fig.tight_layout()
plt.show()

**Как читать этот график:**

Две строки изображений 8x8:
- **Верхняя строка** — зашумлённые изображения, поданные на вход (слева направо: 8 разных цифр).
- **Нижняя строка** — восстановленные версии тех же изображений.

Если автокодировщик справился, нижние изображения выглядят чище и разборчивее верхних. «Горлышко» вынуждено было сохранить только главное — форму цифры — и отбросило случайный шум.

Если восстановление размытое или нечитаемое — попробуйте увеличить `LATENT` (больше информации через горлышко) или число эпох.

### Применение 2. Поиск аномалий

Ключевое свойство: автокодировщик, обученный на «нормальных» объектах, восстанавливает их хорошо (малая ошибка), а незнакомый объект — плохо (большая ошибка). Это и есть принцип детекции аномалий.

В нашем случае «нормальные» — цифры 0–8. Цифра 9 не участвовала в обучении — это «аномалия». Сравним распределения ошибок восстановления для обоих классов.

Порог детекции задаётся по 95-му процентилю ошибки на нормальных данных: всё что выше — аномалия. Выбор процентиля настраивает баланс между пропуском аномалий и ложными тревогами.

In [ ]:
model.eval()
with torch.no_grad():
    rec_normal, _ = model(torch.from_numpy(X_test))
    rec_anom, _ = model(torch.from_numpy(anomaly))

# Ошибка восстановления для каждого объекта.
err_normal = ((rec_normal.numpy() - X_test) ** 2).mean(axis=1)
err_anom = ((rec_anom.numpy() - anomaly) ** 2).mean(axis=1)

# Порог калибруем по нормальным данным (95-й процентиль).
threshold = np.percentile(err_normal, 95)
detected = (err_anom > threshold).mean()

fig, ax = plt.subplots(figsize=(9.5, 5.2))
bins = np.linspace(0, max(err_normal.max(), err_anom.max()), 40)
ax.hist(err_normal, bins=bins, color=VIZ['series'][1], alpha=0.75,
        label='Норма (цифры 0-8)')
ax.hist(err_anom, bins=bins, color=VIZ['series'][2], alpha=0.75,
        label='Аномалия (цифра 9)')
ax.axvline(threshold, color=VIZ['series'][3], linestyle='--',
           label='Порог детекции')
ax.set_title('Распределение ошибки восстановления')
ax.set_xlabel('Ошибка восстановления')
ax.set_ylabel('Число объектов')
ax.legend(loc='upper right')
fig.tight_layout()
plt.show()
print(f'Порог по 95-му процентилю нормы: {threshold:.4f}')
print(f'Доля выявленных аномалий: {detected:.2f}')

**Как читать этот график:**

Гистограммы показывают распределение ошибок восстановления для двух групп объектов:
- **Синяя полоса** — нормальные цифры (0–8): автокодировщик знает их и восстанавливает с малой ошибкой, поэтому гистограмма смещена влево.
- **Оранжевая полоса** — цифра 9 (аномалия): незнакомая сети, восстанавливается хуже, гистограмма смещена вправо.
- **Пунктирная линия** — порог: объекты правее неё помечаются как аномальные.

Хорошая модель: синяя и оранжевая гистограммы должны хорошо разделяться. Если они перекрываются — горлышко слишком широкое (сеть хорошо восстанавливает даже аномалии) или узкое (все восстанавливаются одинаково плохо).

## 5. Интерпретация результата

- **Скрытое представление**: объекты одного класса группируются — автокодировщик без разметки выделил структуру данных. Это нелинейная альтернатива методу главных компонент.
- **Шумоподавление**: восстановленные изображения чище входных. Проходя через узкое «горлышко», сеть передаёт главное и отбрасывает шум.
- **Поиск аномалий**: распределение ошибки для аномального класса сдвинуто вправо. Объекты с ошибкой выше порога помечаются как нетипичные. Порог калибруют по данным **без аномалий**.
- Слишком ёмкое «горлышко» позволяет сети просто копировать вход — тогда ни сжатия, ни выделения признаков не происходит.

Обычный автокодировщик не предназначен для генерации новых объектов: его скрытое пространство не структурировано вероятностно. Для генерации выбирают вариационный автокодировщик или диффузию.

## 6. Попробуйте на своих данных

Автокодировщик применим к изображениям, спектрам, сигналам и табличным данным.

1. Подготовьте матрицу объектов `X` формы `(наблюдение, признак)` со значениями в диапазоне [0, 1].
2. Снимите комментарии в ячейке ниже и подставьте свои данные; при иной размерности входа задайте `n_in` в конструкторе.
3. Для поиска аномалий обучайте сеть только на заведомо нормальных объектах, а порог калибруйте по ним же.
4. Размер «горлышка» `LATENT` подбирайте: слишком малый теряет информацию, слишком большой не сжимает данные.

## 7. Поэкспериментируйте сами

Запустите заново раздел 4, изменив один из параметров ниже, и сравните результаты:

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `LATENT = 2` | Уменьшить горлышко до 2 | Кривая обучения сходится медленнее, восстановление размытее — но карта скрытого пространства будет 2D без PCA |
| `LATENT = 32` | Увеличить горлышко до 32 | Восстановление чище, но кластеры на карте хуже разделены — сеть «ленится» сжимать |
| `noise = 0.5` | Удвоить уровень шума при шумоподавлении | При каком уровне шума сеть перестаёт справляться? |
| `threshold percentile = 99` | В ячейке аномалий: `np.percentile(err_normal, 99)` | Порог сдвигается правее — меньше ложных тревог, но и меньше находок |
| `n_epochs = 120` | Удвоить число эпох | Улучшается ли детекция аномалий при более глубоком обучении? |

Совет: после изменения `LATENT` перезапускайте обе ячейки — архитектуры и обучения.

In [ ]:
# --- Шаблон загрузки своих данных ---
# import pandas as pd
#
# X = pd.read_csv('my_data.csv').to_numpy(dtype='float32')
# X = (X - X.min(0)) / (X.max(0) - X.min(0) + 1e-8)
#
# print(f'Объектов: {len(X)}, признаков: {X.shape[1]}')
# При смене размерности входа задайте n_in в конструкторе Autoencoder.

## 8. Краткие выводы

- Автокодировщик — нейронная сеть, обучаемая без разметки воспроизводить свой вход через узкое «горлышко».
- Скрытый вектор (latent vector) в горлышке — компактное нелинейное представление данных, пригодное для визуализации и поиска похожих объектов.
- Шумоподавление достигается автоматически: горлышко пропускает устойчивые признаки и отфильтровывает шум.
- Поиск аномалий работает через ошибку восстановления: нормальные объекты восстанавливаются хорошо, нетипичные — плохо; порог калибруется по нормальным данным.
- Слишком широкое горлышко устраняет эффект сжатия; слишком узкое ухудшает восстановление — ширину подбирают экспериментально.
- Для генерации новых объектов обычный автокодировщик не подходит: его скрытое пространство не структурировано вероятностно. Для генерации используют вариационный автокодировщик (VAE) или диффузионные модели.